In [ ]:
from google.cloud import bigquery

client = bigquery.Client(project="prc-data-challenge-2025")


def q(sql):
    return client.query(sql).to_dataframe()


# preview 10 rows
q("""
SELECT *
FROM `prc-data-challenge-2025.prc2025.fuel_train`
LIMIT 10
""")

fuel_train: one row is one interval of a flight (about 12 intervals per flight on average).

idx is the interval id, unique for each row. flight_id is the flight id, shared by all intervals of the same flight; it is the key to join with the other tables.

start and end are the interval bounds, stored as integers counting nanoseconds since 1970-01-01 UTC. We know it because they have 19 digits, and converting them gives dates in April 2025 with intervals of 5 to 30 minutes.

fuel_kg is the fuel burnt during the interval, in kg. It is the target to predict.

In [ ]:
# check: start/end are nanoseconds
q("""
SELECT
  flight_id,
  TIMESTAMP_MICROS(DIV(start, 1000)) AS start_ts,
  TIMESTAMP_MICROS(DIV(`end`, 1000)) AS end_ts,
  (`end` - start) / 1e9 AS duration_s,
  fuel_kg
FROM `prc-data-challenge-2025.prc2025.fuel_train`
LIMIT 10
""")

Same df but with the start and end columns converted to timestamps and a new column for duration in seconds.

In [ ]:
# global counts and period
q("""
SELECT
  COUNT(*) AS n_intervals,
  COUNT(DISTINCT flight_id) AS n_flights,
  COUNT(*) / COUNT(DISTINCT flight_id) AS avg_intervals_per_flight,
  MIN(TIMESTAMP_MICROS(DIV(start, 1000))) AS first_start,
  MAX(TIMESTAMP_MICROS(DIV(`end`, 1000))) AS last_end,
  AVG((`end` - start) / 1e9) AS avg_duration_s,
  AVG(fuel_kg) AS avg_fuel_kg
FROM `prc-data-challenge-2025.prc2025.fuel_train`
""")

Some key number to get a better representation of the data to predict.
There is 131530 part of flight, 11037 unique fligts.
The avg interval ~12 across the dataset for 1 flight
The data set timeline is 2025-04-12 -> 2025-09-01.
The average duration of an interval = 531s.
The average consumtion of fuel per fligh part is 557kg.

In [ ]:
# distributions and invalid values
q("""
SELECT
  APPROX_QUANTILES((`end` - start) / 1e9, 4) AS duration_s_quartiles,
  APPROX_QUANTILES(fuel_kg, 4) AS fuel_kg_quartiles,
  COUNTIF(`end` <= start) AS n_non_positive_duration,
  COUNTIF(fuel_kg IS NULL) AS n_null_fuel,
  COUNTIF(fuel_kg <= 0) AS n_non_positive_fuel
FROM `prc-data-challenge-2025.prc2025.fuel_train`
""")

No null or negative values, but some extremes are suspicious.
Interval duration: min 1.61 s, max 3600 s. Half of the intervals last under 300 s, half more; the middle half lasts between 5 and 10 min.
fuel_kg per interval: min 0.45 kg (= 1 lb), max 32205 kg (= 71000 lb). Half of the intervals burn under 200 kg, half more. Q3 = 544.31 kg (= 1200 lb).
-> part of the data was measured in pounds then converted to kg: several data sources with different rounding.

In [ ]:
# most extreme fuel rates
q("""
SELECT
  idx,
  flight_id,
  (`end` - start) / 1e9 AS duration_s,
  fuel_kg,
  fuel_kg / ((`end` - start) / 1e9) AS fuel_rate_kg_s
FROM `prc-data-challenge-2025.prc2025.fuel_train`
ORDER BY fuel_rate_kg_s DESC
LIMIT 20
""")

Top 20 fuel rates (kg/s): 12 to 16 kg/s, far above a real airliner (~2-3 kg/s cruise, ~4-5 kg/s takeoff).
Two kinds of extremes:
- very short intervals (7 s to 150 s) with round values (100, 400, 900, 2500 kg): rounding artefact of the fuel gauge (100 kg steps), not physically meaningful but very noisy.
- whole flights with high rates on every interval, values round in pounds (7257 kg = 16000 lb, 9071 kg = 20000 lb, 32205 kg = 71000 lb): systematic issue for these flights (unit error? other measure? aircraft type?).
Suspect flights to check with flightlist_train: prc790175280, prc784170877, prc783282112, prc783379216.

In [ ]:
# average fuel rate per flight
df_rate = q(
    """
SELECT flight_id, COUNT(*) AS n_intervals,
SUM(fuel_kg) / SUM((`end` - start) / 1e9) AS avg_fuel_rate_kg_s
FROM `prc-data-challenge-2025.prc2025.fuel_train`
GROUP BY flight_id
"""
)

In [ ]:
# average fuel rate distribution
df_rate.describe()

Average fuel rate per flight (sum fuel / sum interval duration):
median 0.65 kg/s (~ A320 cruise, 57% of flights), Q1 0.45, Q3 1.36 (widebodies, intervals include descent), max 13.1.
-> high outliers: flights > 6 kg/s are impossible (to count).
-> low outliers too: min 0.04 kg/s (~150 kg/h), below the idle burn of a twin jet (~0.2 kg/s) -> suspicious.